# PROJECT STAGE

Current stage: Research exploration

Code may later be migrated to src modules once stabilized.

# 1. Metadata

In [ ]:
# ============================================================
# NOTEBOOK METADATA
# ============================================================

NOTEBOOK_NAME = "03_systemic_pipeline_v1"
AUTHOR = "Juan Manuel Giacame"
CREATED = "2026-03-29"
UPDATED = "2026-03-29"
GIT_HASH = "..."

DESCRIPTION = """ 
 Configuration-driven pipeline for building systemic (cross-asset) features from panel data.

This notebook orchestrates panel construction, systemic feature aggregation, validation, and versioned export, serving as the foundation for regime detection and portfolio research workflows."""

# 2. Imports


In [ ]:
import pandas as pd
import plotly.express as px

from quant_research.data.registry.universe_registry import get_all_assets
from quant_research.panels.builders.panel_builder import PanelBuilder
from quant_research.features.catalog.feature_catalog_builder import FeatureCatalogBuilder
from quant_research.systemic.preprocessing.feature_resolver import FeatureResolver
from quant_research.systemic.preprocessing.panel_preparer import PanelPreparer
from quant_research.systemic.builders.systemic_builder import SystemicBuilder

from quant_research.features.loaders.asset_feature_loader import FeatureLoader
from quant_research.data.processed.loaders.asset_processed_loader import AssetProcessedDataLoader #ver si se puede eliminar

from quant_research.config.paths import SYSTEMIC_PATH, PROCESSED_DATA_PATH, ASSET_FEATURE_PATH

# 3. CONFIGURATION


In [ ]:
# ============================================================
# SYSTEMIC RESEARCH CONFIGURATION
# ============================================================

CONFIG = {

    # ========================================================
    # PANEL CONFIGURATION (DATA LOADING & STRUCTURE)
    # ========================================================
    "panel": {

        # ----------------------------------------------------
        # ASSET UNIVERSE
        # ----------------------------------------------------
        "assets": [a.symbol for a in get_all_assets()],
		
		# [
            # Example:
            # "SPY", "QQQ", "TLT", "GLD", "BTC", "ETH"
        # ],

        # ----------------------------------------------------
        # FEATURE FAMILIES TO LOAD
        # ----------------------------------------------------
        # Defines which asset-level feature families are loaded
        # into the panel (must exist in feature pipeline output)
        "families": [
            "momentum",
            "volatility",
            # future:
            # "liquidity", "carry", "seasonality"
        ],

        # ----------------------------------------------------
        # PANEL ALIGNMENT METHOD
        # ----------------------------------------------------
        # "union"        → keeps full timeline (recommended)
        # "intersection" → keeps only common timestamps
        "alignment": "union",

        # ----------------------------------------------------
        # NaN HANDLING CONFIGURATION
        # ----------------------------------------------------
        "nan_handling": {

            # --------------------------------------------
            # WARMUP HANDLING
            # --------------------------------------------
            "warmup": {

                # "cut"  → remove initial periods with missing data
                # "keep" → keep full history (NaNs allowed)
                "method": "cut",

                # "used_features" → based only on features used in systemic
                # "all"           → based on entire panel
                "based_on": "used_features",
            },

            # --------------------------------------------
            # CALENDAR ALIGNMENT
            # --------------------------------------------
            "calendar": {

                # "ffill" → forward fill (recommended in finance)
                # "bfill" → backward fill
                # "drop"  → drop rows with any NaN
                # "none"  → no action
                "method": "ffill",
            },

            # --------------------------------------------
            # FINAL CLEANING STEP
            # --------------------------------------------
            "final": {

                # "dropna" → remove any remaining NaNs
                # "keep"   → allow NaNs in final dataset
                "method": "dropna",
            }
        },
    },

    # ========================================================
    # SYSTEMIC FEATURE CONFIGURATION (CORE RESEARCH LAYER)
    # ========================================================
    "systemic": {

        "features": [

            # ====================================================
            # 1. MEAN (cross-asset average)
            # ====================================================
            {
                "name": "mean_{feature}_{window}",
                "type": "mean",

                # Input features (must exist in feature catalog)
                "features": ["{feature}_{window}"],

                # Parameter expansion (grid search style)
                "expand": {
                    "feature": ["MOM"],   # feature families
                    "window": [5, 21, 63],       # lookback horizons
                }
            },

            # ====================================================
            # 2. MEAN (Ret _1 cross-asset average)
            # ====================================================
            {
                "name": "mean_{feature}_{window}",
                "type": "mean",

                # Input features (must exist in feature catalog)
                "features": ["{feature}_{window}"],

                # Parameter expansion (grid search style)
                "expand": {
                    "feature": ["RET"],   # feature families
                    "window": [1, 5, 21, 63],       # lookback horizons
                }
            },
            # ====================================================
            # 3. MEAN_VOL_Z (cross-asset average)
            # ====================================================
            {
                "name": "mean_{feature}_{window}",
                "type": "mean",

                # Input features (must exist in feature catalog)
                "features": ["{feature}_{window}_Z"],

                # Parameter expansion (grid search style)
                "expand": {
                    "feature": ["VOL"],   # feature families
                    "window": [5, 21, 63],       # lookback horizons
                }
            },
            # ====================================================
            # 4. DISPERSION (cross-asset std)
            # ====================================================
            {
                "name": "dispersion_{feature}_{window}",
                "type": "dispersion",

                "features": ["{feature}_{window}_Z"],

                "expand": {
                    "feature": ["MOM"],
                    "window": [5, 21, 63],
                },

                # Optional parameters for aggregator
                "params": {
                    # "method": "std"
                }
            },

            # ====================================================
            # 3. BREADTH (fraction of assets above threshold)
            # ====================================================
            {
                "name": "breadth_{feature}_{window}",
                "type": "breadth",

                "features": ["{feature}_{window}_Z"],

                "expand": {
                    "feature": ["MOM"],
                    "window": [5, 21, 63],
                },

                "params": {
                    "threshold": 0.0   # e.g. positive momentum
                }
            },

            # ====================================================
            # 4. CORRELATION (market synchronization)
            # ====================================================
            {
                "name": "corr_{feature}_{window}_{corr_window}",
                "type": "correlation",

                "features": ["{feature}_{window}_Z"],

                "expand": {
                    "feature": ["MOM"],
                    "window": [5, 21, 63],
                    "corr_window": [10, 20, 63],
                },

                "params": {
                    "window": "{corr_window}"
                }
            },

            # ====================================================
            # 5. Z-SCORE TRANSFORM (on systemic outputs)
            # ====================================================
            {
                "name": "mean_{feature}_{window}_z_{z_window}",
                "type": "zscore",

                # Uses output of another systemic feature
                "input": "mean_{feature}_{window}",

                "expand": {
                    "feature": ["MOM"],
                    "window": [5, 21, 63],
                    "z_window": [20, 63],
                },

                "params": {
                    "window": "{z_window}"
                }
            },

            # ====================================================
            # 6. RANGE (cross-asset max-min)
            # ====================================================
            {
                "name": "range_{feature}_{window}",
                "type": "range",

                "features": ["{feature}_{window}_Z"],

                "expand": {
                    "feature": ["MOM"],
                    "window": [5, 21, 63],
                }
            },
        ]
    },

    # ========================================================
    # OUTPUT CONFIGURATION
    # ========================================================
    "output": {

        # Base path for saving systemic datasets
        "base_path": SYSTEMIC_PATH,  # set externally (e.g. SYSTEMIC_PATH)

        # Run identifier (used for versioning experiments)
        "run_name": "baseline_v1",

        # Future:
        # "versioning": "timestamp" | "manual"
    }
}

# 4. Data Loading

In [ ]:
feature_loader = FeatureLoader()
processed_loader = AssetProcessedDataLoader()
processed_path = PROCESSED_DATA_PATH /"metadata.json"

panel_builder = PanelBuilder(
    feature_loader=feature_loader,
    processed_loader=processed_loader,
    processed_metadata_path=processed_path
)

panel = panel_builder.build_panel(
    assets=CONFIG["panel"]["assets"],
    families=["momentum", "volatility"],  # o dinámico en el futuro
    alignment=CONFIG["panel"]["alignment"],
    nan_policy="keep",  # importante: el preparer decide después
)

In [ ]:
feature_path= ASSET_FEATURE_PATH /"registry_snapshot.json"

catalog_builder = FeatureCatalogBuilder(processed_metadata_path=processed_path, registry_snapshot_path=feature_path)

catalog =catalog_builder.build()

In [ ]:
from quant_research.systemic.config.config_expander import SystemicConfigExpander
builder = SystemicConfigExpander(CONFIG["systemic"]["features"])
features = builder.expand()

In [ ]:
resolver = FeatureResolver(catalog)
required_features = resolver.resolve(features)

In [ ]:
preparer = PanelPreparer(CONFIG["panel"]["nan_handling"])

panel_prepared, meta = preparer.prepare(
    panel=panel,
    required_features=required_features
)

## 4.1 Panel data audit

In [ ]:
import pandas as pd
import numpy as np

def audit_panel(panel: pd.DataFrame):
    
    assert isinstance(panel.columns, pd.MultiIndex), "Panel must have MultiIndex columns"

    features = panel.columns.get_level_values(0).unique()
    assets = panel.columns.get_level_values(1).unique()

    report = {}

    # ============================================================
    # SUMMARY
    # ============================================================

    report["summary"] = {
        "shape": panel.shape,
        "n_features": len(features),
        "n_assets": len(assets),
        "start": str(panel.index.min()),
        "end": str(panel.index.max()),
        "n_dates": panel.index.nunique(),
    }

    # ============================================================
    # NaNs (GLOBAL)
    # ============================================================

    total_nans = panel.isna().sum().sum()
    total_values = panel.size

    report["summary"]["nan_ratio"] = float(total_nans / total_values)

    # ============================================================
    # PER FEATURE
    # ============================================================

    per_feature = []

    for feat in features:
        df_feat = panel[feat]

        nan_ratio = df_feat.isna().sum().sum() / df_feat.size

        per_feature.append({
            "feature": feat,
            "nan_ratio": float(nan_ratio),
            "mean": float(df_feat.mean().mean()),
            "std": float(df_feat.std().mean()),
            "min": float(df_feat.min().min()),
            "max": float(df_feat.max().max()),
        })

    report["per_feature"] = pd.DataFrame(per_feature)

    # ============================================================
    # PER ASSET
    # ============================================================

    per_asset = []

    for asset in assets:
        df_asset = panel.xs(asset, level=1, axis=1)

        nan_ratio = df_asset.isna().sum().sum() / df_asset.size

        per_asset.append({
            "asset": asset,
            "nan_ratio": float(nan_ratio),
            "mean": float(df_asset.mean().mean()),
            "std": float(df_asset.std().mean()),
        })

    report["per_asset"] = pd.DataFrame(per_asset)

    # ============================================================
    # COVERAGE CHECK
    # ============================================================

    coverage = panel.notna().T.groupby(level="feature").mean().T

    report["coverage"] = coverage

    # ============================================================
    # OUTLIER CHECK (simple z-score)
    # ============================================================

    z = (panel - panel.mean()) / panel.std()
    extreme = (np.abs(z) > 5).sum().sum()

    report["summary"]["extreme_values"] = int(extreme)

    return report

In [ ]:
audit = audit_panel(panel)
audit["summary"]

In [ ]:
audit["per_feature"].sort_values("nan_ratio", ascending=False)

In [ ]:
audit["per_asset"].sort_values("nan_ratio", ascending=False)

In [ ]:
audit["coverage"].tail()

## 4.2 Panel_prepared data audit

In [ ]:
audit = audit_panel(panel_prepared)
audit["summary"]

# 5. Core Computations

## 5.1 Compute Systemic Features

In [ ]:
from quant_research.systemic.orchestrator_v1 import SystemicOrchestrator

orchestrator = SystemicOrchestrator(CONFIG)

bundle = orchestrator.run(panel_prepared=panel_prepared,  output_path=SYSTEMIC_PATH / CONFIG["output"]["run_name"])

df_systemic = bundle.data
df_systemic_z = bundle.data_z
meta = bundle.metadata

In [ ]:
for col in df_systemic.columns:
    print(col, df_systemic[col].notna().sum())

In [ ]:
if CONFIG["panel"]["nan_handling"]["final"]["method"] == "dropna":
    df_systemic = df_systemic.dropna()

In [ ]:
df_systemic.head()

# 6. Systemic Dataset Audit

In [ ]:
import pandas as pd
import numpy as np

def audit_systemic(df: pd.DataFrame):
    
    report = {}

    # ============================================================
    # BASIC STRUCTURE
    # ============================================================

    assert isinstance(df, pd.DataFrame), "Systemic must be DataFrame"
    assert not df.empty, "Systemic DataFrame is empty"
    assert isinstance(df.index, pd.DatetimeIndex), "Index must be DatetimeIndex"
    assert df.index.is_monotonic_increasing, "Index must be sorted"

    report["shape"] = df.shape
    report["start"] = str(df.index.min())
    report["end"] = str(df.index.max())

    # ============================================================
    # NaNs (STRICT)
    # ============================================================

    total_nans = df.isna().sum().sum()
    assert total_nans == 0, f"Systemic contains NaNs: {total_nans}"

    report["nan_check"] = "passed"

    # ============================================================
    # DUPLICATES
    # ============================================================

    assert not df.index.duplicated().any(), "Duplicate timestamps found"
    report["duplicates"] = "none"

    # ============================================================
    # CONSTANT FEATURES
    # ============================================================

    constant_cols = [col for col in df.columns if df[col].std() == 0]

    assert len(constant_cols) == 0, f"Constant features found: {constant_cols}"

    report["constant_features"] = "none"

    # ============================================================
    # EXTREME VALUES
    # ============================================================

    z = (df - df.mean()) / df.std()
    extreme = (np.abs(z) > 8).sum().sum()

    report["extreme_values"] = int(extreme)

    # no assert → solo warning
    if extreme > 0:
        print(f"⚠️ Warning: {extreme} extreme values detected")

    # ============================================================
    # VALUE RANGES (sanity)
    # ============================================================

    ranges = {}

    for col in df.columns:
        ranges[col] = {
            "min": float(df[col].min()),
            "max": float(df[col].max())
        }

    report["ranges"] = ranges

    # ============================================================
    # FINAL STATUS
    # ============================================================

    report["status"] = "PASSED"

    return report

In [ ]:
audit_sys = audit_systemic(df_systemic)
audit_sys

# 7. Export

## 7.1 Systemic Dataset export audit

In [ ]:
output_path=SYSTEMIC_PATH / CONFIG["output"]["run_name"]

# ------------------------------------------------
# LOAD BACK
# ------------------------------------------------

df_loaded = pd.read_parquet(output_path / "systemic.parquet")
df_loaded_z = pd.read_parquet(output_path / "systemic_z.parquet")

# ------------------------------------------------
# VALIDATE
# ------------------------------------------------

assert np.allclose(df_loaded.values, df_systemic.values, equal_nan=True)
assert list(df_loaded.columns) == list(df_systemic.columns)
assert df_loaded.index.equals(df_systemic.index)

assert np.allclose(df_loaded_z.values, df_systemic_z.values, equal_nan=True)

In [ ]:
print("✅ Export completed")
print(f"📁 Path: {output_path}")

print("\nShape:", df_systemic.shape)

print("\nNaN ratio:")
print(df_systemic.isna().mean())

print("\nPreview:")
print(df_systemic.head())

# 8. Visualization

In [ ]:
spy_price = panel["PRICE"]["SPY"]
spy_price = spy_price.loc[df_systemic.index]

In [ ]:
df_z = df_systemic_z

In [ ]:
spy_z = (spy_price - spy_price.mean()) / spy_price.std()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

features = df_z.columns
n = len(features)

fig = make_subplots(
    rows=n,
    cols=1,
    shared_xaxes=False,
    subplot_titles=[f"{col} vs SPY (Z)" for col in features]
)

for i, col in enumerate(features, start=1):
    
    # feature Z
    fig.add_trace(
        go.Scatter(
            x=df_z.index,
            y=df_z[col],
            name=col,
            line=dict(width=2)
        ),
        row=i, col=1
    )
    
    # SPY Z
    fig.add_trace(
        go.Scatter(
            x=spy_z.index,
            y=spy_z,
            name="SPY_Z",
            line=dict(dash="dot"),
            opacity=0.7
        ),
        row=i, col=1
    )

fig.update_layout(
    height=300 * n,
    title="Systemic Features (Z) vs SPY (Z)",
    showlegend=False,
    template = "plotly_dark"
)
fig.update_xaxes(
    dtick="M3",   # cada 3 meses
)

fig.show()